In [ ]:
!pip install ultralytics torchreid

# Project dependencies (imports)

In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
from sklearn.cluster import KMeans
import cv2
from cv2.typing import MatLike
import torch
import torchreid
from torchvision import transforms
from ultralytics import YOLO
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Configs

In [10]:
IMAGES_FOLDER = Path('/kaggle/input/datasets/davideperniconi/samples-videos/train/SNGS-060/img1')

VIDEO_DURATION = 30
VIDEO_OUTPUT = Path('/kaggle/working/tracked.mp4')
VIDEO_PATH = Path("/kaggle/input/football-soccer-videos-dataset/107.mp4")

DETECTION_BATCH_SIZE = 15
FPS = 25.0

YOLO_WEIGHTS_PATH = Path('/kaggle/input/models/stefanopassanante/soccermodel/other/default/1/best.pt')

# Models (YOLO + ReID)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(YOLO_WEIGHTS_PATH)

reid_model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=1000, 
    pretrained=True
).to(device)

reid_model.eval()

reid_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Utils

In [16]:
def read_images(folder_path: Path):
    path = folder_path / '*.jpg'
    match = glob.glob(str(path))

    files = sorted(match)
    frames = []

    for f in tqdm(files, desc="Reading images"):
        frame = cv2.imread(f)

        if frame is None:
            print(f"Impossible to read: {f}")
        else:
            frames.append(frame)

    print(f'Acquired {len(frames)} images from {folder_path}')
    
    return frames

def video_to_frames(video_path: Path, frames_dir: Path, expected_fps: float = None):
    video_path = Path(video_path)
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 0.0

    if expected_fps is not None and fps > 0 and abs(expected_fps - fps) > 0.5:
        print(f"FPS mismatch: expected {expected_fps:.2f}, actual {fps:.2f}. Userò quello reale ({fps:.2f}).")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

    i = 0
    pbar = tqdm(total=total_frames, desc="Extracting frames")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        out = frames_dir / f"{i:06d}.jpg"
        cv2.imwrite(str(out), frame)
        i += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    return fps, i  


def save_as_video(frames: list[MatLike], duration: int, path: Path, fps: float = 25.0):
   
    if path is None:
        raise ValueError("Output path is None")
    
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    
    if len(frames) == 0:
        raise ValueError("No frames to save")
    
    h, w = frames[0].shape[:2]
    
    required_frames = int(duration * fps)
    
    if len(frames) != required_frames:
        print(f"Info: {len(frames)} frame disponibili, {required_frames} richiesti per {duration}s a {fps}fps")
        if len(frames) < required_frames:
            frames = frames + [frames[-1]] * (required_frames - len(frames))
        else:
            frames = frames[:required_frames]
    
    print(f"Salvataggio video: {len(frames)} frame, {w}x{h}, {fps}fps")
    print(f"Output: {path}")
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(path), fourcc, fps, (w, h))
    
    if not writer.isOpened():
        raise RuntimeError(f"cv2.VideoWriter non può essere aperto. path={path}")
    
    for f in tqdm(frames, desc="Writing video"):
        if f.shape[0] != h or f.shape[1] != w:
            f = cv2.resize(f, (w, h))
        writer.write(f)
    
    writer.release()
    print(f"Video salvato: {path}")



## Updated Reid Multi embedding

In [17]:

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0, inter_x2 - inter_x1)
    ih = max(0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

class OcclusionSwapGuard:
    def __init__(self, iou_cluster=0.2, iou_prev_match=0.3, sim_high=0.82, gap_low=0.08):
        self.iou_cluster = iou_cluster
        self.iou_prev_match = iou_prev_match
        self.sim_high = sim_high
        self.gap_low = gap_low
        self.prev_boxes = None
        self.prev_ids = None
        self.prev_clusters = None


    def _clusters_from_boxes(self, boxes):
        n = len(boxes)
        parent = list(range(n))
    
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
    
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    
        for i in range(n):
            for j in range(i + 1, n):
                if iou_xyxy(boxes[i], boxes[j]) >= self.iou_cluster:
                    union(i, j)
    
        comps = {}
        for i in range(n):
            r = find(i)
            comps.setdefault(r, []).append(i)
    
        return list(comps.values())


    def guard(self, boxes, proposed_ids, reid_sims=None, reid_gaps=None):
        n = len(proposed_ids)
        if n == 0:
            self.prev_boxes = boxes
            self.prev_ids = proposed_ids
            self.prev_clusters = []
            return proposed_ids, []

        if reid_sims is None:
            reid_sims = [0.0] * n
        if reid_gaps is None:
            reid_gaps = [0.0] * n

        clusters = self._clusters_from_boxes(boxes)
        final_ids = list(proposed_ids)
        alerts = []

        if self.prev_boxes is None or self.prev_ids is None:
            self.prev_boxes = boxes.copy()
            self.prev_ids = list(final_ids)
            self.prev_clusters = clusters
            return final_ids, alerts

        prev_best = [-1] * n
        prev_best_iou = [0.0] * n
        for i in range(n):
            best_j = -1
            best_i = 0.0
            for j in range(len(self.prev_boxes)):
                v = iou_xyxy(boxes[i], self.prev_boxes[j])
                if v > best_i:
                    best_i = v
                    best_j = j
            prev_best[i] = best_j
            prev_best_iou[i] = best_i

        for idxs in clusters:
            if len(idxs) <= 1:
                continue

            for i in idxs:
                j = prev_best[i]
                if j == -1 or prev_best_iou[i] < self.iou_prev_match:
                    continue

                prev_id = self.prev_ids[j]
                cur_id = proposed_ids[i]

                if cur_id != prev_id:
                    is_reid_confident = (reid_sims[i] >= self.sim_high) and (reid_gaps[i] >= self.gap_low)

                    if not is_reid_confident:
                        final_ids[i] = prev_id
                        alerts.append({
                            "type": "possible_swap_corrected",
                            "det_index": int(i),
                            "prev_id": int(prev_id),
                            "proposed_id": int(cur_id),
                            "prev_iou": float(prev_best_iou[i]),
                            "reid_sim": float(reid_sims[i]),
                            "reid_gap": float(reid_gaps[i]),
                        })
                    else:
                        alerts.append({
                            "type": "swap_allowed_reid_confident",
                            "det_index": int(i),
                            "prev_id": int(prev_id),
                            "proposed_id": int(cur_id),
                            "prev_iou": float(prev_best_iou[i]),
                            "reid_sim": float(reid_sims[i]),
                            "reid_gap": float(reid_gaps[i]),
                        })

        self.prev_boxes = boxes.copy()
        self.prev_ids = list(final_ids)
        self.prev_clusters = clusters
        return final_ids, alerts



In [26]:
class Detection:
    def __init__(self, yolo):
        self.yolo = yolo

    def process_frame(self, frame):
        results = self.yolo.track(
            frame,
            persist=True,
            tracker="project_tracker.yaml",
            conf=0.31,
            verbose=False
        )

        if results is None or len(results) == 0:
            return None, None, None

        result = results[0]

        if result.boxes is None:
            return None, None, None

        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)

        if result.boxes.id is None:
            ids = np.zeros(len(boxes), dtype=int)
        else:
            ids = result.boxes.id.cpu().numpy().astype(int)

        return boxes, ids, classes        

class ReID:
    def __init__(
        self,
        reid_model,
        reid_transforms,
        sim_match: float = 0.75,
        sim_add: float = 0.50,
        max_embeddings: int = 6,
        alpha: float = 0.2
    ):
        self.reid_model = reid_model
        self.reid_transforms = reid_transforms
        self.device = next(reid_model.parameters()).device

        self.memory = {} 
        self.sim_match = sim_match
        self.sim_add = sim_add
        self.max_embeddings = max_embeddings
        self.alpha = alpha

    def to_embeddings(self, crops: list):
        if not crops: return np.array([])
        batch = torch.stack([self.reid_transforms(c) for c in crops]).to(self.device)
        with torch.inference_mode():
            embeddings = self.reid_model(batch).cpu().numpy()
        return embeddings

    @staticmethod
    def _cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
        return 1.0 - cosine(a, b)

    def update_frame(self, current_embeddings, current_yolo_ids, update_memory: bool = True):
        if len(current_embeddings) == 0:
            return []

        gallery_ids = list(self.memory.keys())
        
        if not gallery_ids:
            for i, emb in enumerate(current_embeddings):
                self._append_embedding(current_yolo_ids[i], emb)
            return list(current_yolo_ids)

        num_det = len(current_embeddings)
        num_gal = len(gallery_ids)
        cost_matrix = np.zeros((num_det, num_gal))

        for i in range(num_det):
            for j in range(num_gal):
                g_id = gallery_ids[j]
                sims = [self._cosine_sim(current_embeddings[i], ge) for ge in self.memory[g_id]]
                cost_matrix[i, j] = 1.0 - max(sims)

        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        final_ids = [None] * num_det
        assigned_indices = set()

        
        for r, c in zip(row_ind, col_ind):
            cost = cost_matrix[r, c]
            actual_id = gallery_ids[c]

            if (1.0 - cost) >= self.sim_match:
                final_ids[r] = actual_id
                if update_memory:
                    self._update_existing_id(actual_id, current_embeddings[r])
                assigned_indices.add(r)

        for i in range(num_det):
            if i not in assigned_indices:
                yolo_id = current_yolo_ids[i]
                final_ids[i] = yolo_id
                if update_memory:
                    self._append_embeddingUpdated(yolo_id, current_embeddings[i])

        return final_ids

    def assign_with_scores(self, current_embeddings, current_yolo_ids, update_memory: bool = False):
        if len(current_embeddings) == 0:
            return [], [], []
    
        gallery_ids = list(self.memory.keys())
        num_det = len(current_embeddings)
    
        if not gallery_ids:
            final_ids = list(current_yolo_ids)
            best_sims = [1.0] * num_det
            gaps = [1.0] * num_det
            if update_memory:
                for i, emb in enumerate(current_embeddings):
                    self._append_embedding(current_yolo_ids[i], emb)
            return final_ids, best_sims, gaps
    
        num_gal = len(gallery_ids)
        cost_matrix = np.zeros((num_det, num_gal), dtype=np.float32)
        sim_matrix = np.zeros((num_det, num_gal), dtype=np.float32)
    
        for i in range(num_det):
            for j in range(num_gal):
                g_id = gallery_ids[j]
                sims = [self._cosine_sim(current_embeddings[i], ge) for ge in self.memory[g_id]]
                s = float(max(sims))
                sim_matrix[i, j] = s
                cost_matrix[i, j] = 1.0 - s
    
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
        final_ids = [None] * num_det
        best_sims = [0.0] * num_det
        gaps = [0.0] * num_det
        assigned = set()
    
        for i in range(num_det):
            sims = sim_matrix[i]
            if len(sims) >= 2:
                top2 = np.partition(sims, -2)[-2:]
                s1, s2 = float(np.max(top2)), float(np.min(top2))
                gaps[i] = s1 - s2
            elif len(sims) == 1:
                gaps[i] = 1.0
    
        for r, c in zip(row_ind, col_ind):
            s = float(sim_matrix[r, c])
            best_sims[r] = s
            actual_id = gallery_ids[c]
    
            if s >= self.sim_match:
                final_ids[r] = actual_id
                if update_memory:
                    self._update_existing_id(actual_id, current_embeddings[r])
                assigned.add(r)
    
        for i in range(num_det):
            if i not in assigned:
                final_ids[i] = int(current_yolo_ids[i])
                best_sims[i] = float(np.max(sim_matrix[i]))  # “quanto era vicino” al gallery, anche se non matchato
                if update_memory:
                    self._append_embeddingUpdated(final_ids[i], current_embeddings[i])
    
        return final_ids, best_sims, gaps


    def _update_existing_id(self, track_id, embedding):
        sims = [self._cosine_sim(embedding, e) for e in self.memory[track_id]]
        best_idx = np.argmax(sims)
        self.memory[track_id][best_idx] = (1 - self.alpha) * self.memory[track_id][best_idx] + self.alpha * embedding
        if max(sims) < self.sim_add:
            self._append_embeddingUpdated(track_id, embedding)

    
    def _append_embedding(self, track_id, embedding):
        if track_id not in self.memory: self.memory[track_id] = []
        self.memory[track_id].append(embedding)
        if len(self.memory[track_id]) > self.max_embeddings:
            self.memory[track_id].pop(0)

    def _append_embeddingUpdated(self, track_id, embedding):
        if track_id not in self.memory:
            self.memory[track_id] = []
    
        self.memory[track_id].append(embedding)
    
        if self.max_embeddings is None:
            return
            
        while len(self.memory[track_id]) > self.max_embeddings:
            embs = self.memory[track_id]
            n = len(embs)
    
            max_sim_to_others = []
            for i in range(n):
                sims_i = []
                for j in range(n):
                    if i == j:
                        continue
                    sims_i.append(self._cosine_sim(embs[i], embs[j]))
                max_sim_to_others.append(max(sims_i))
    
            remove_idx = int(np.argmax(max_sim_to_others))
            embs.pop(remove_idx)


class Annotator:
    def __init__(self):
        pass  
    
    def player_bounding_box(self, frame, box, obj_id, confidence: float = 1.0):
        
        x1, y1, x2, y2 = map(int, box)
        
        color = (0, 255, 0)
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        label = f"ID {obj_id}"
        cv2.putText(frame, label, (x1, max(0, y1-10)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        return frame
    
    def highlight_ball(self, frame, ball_box, ball_id=0):
       
        if ball_box is None:
            return frame
            
        x1, y1, x2, y2 = map(int, ball_box)
        
        color = (0, 255, 255)
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        cv2.putText(frame, "BALL", (x1, max(0, y1-10)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        return frame

   

class VideoProcessor:
    def __init__(
        self,
        detection: Detection,
        reid: ReID,
        annotator: Annotator = None,
        occlusion_guard: OcclusionSwapGuard = None,
        possession_threshold: int = 75
    ):
        self.detection = detection
        self.reid = reid
        self.annotator = annotator
        self.occlusion_guard = occlusion_guard
        self.possession_threshold = possession_threshold

        self.ball_positions_history = []
        self.players_positions_history = []
        self.possession_dict = {}

        self.BALL_CLASS = 0
        self.LEFT_CLASSES = {1, 3}
        self.RIGHT_CLASSES = {2, 4}

        self.team_in_possession_history = []

    def process(self, frames: list[MatLike]):
        all_detections = []
        for frame_idx, frame in tqdm(enumerate(frames), desc="Processing frames"):
            frame_detections = self.process_frame(frame)
            all_detections.append(frame_detections)
        return all_detections

    def process_frame(self, frame: MatLike):
        boxes, ids, classes = self.detection.process_frame(frame)

        current_frame_detections = []
        current_ball_pos = None
        current_frame_players = []

        if boxes is not None:
            valid_crops = []
            valid_indices = []
            valid_yolo_ids = []

            for j, box in enumerate(boxes):
                if classes[j] == 5:
                    continue

                if classes[j] == self.BALL_CLASS:
                    x1, y1, x2, y2 = map(int, box)
                    current_ball_pos = [(x1 + x2) / 2, (y1 + y2) / 2]

                    detection_tuple = (
                        int(ids[j]),
                        float(x1),
                        float(y1),
                        float(x2 - x1),
                        float(y2 - y1),
                        int(classes[j])
                    )
                    current_frame_detections.append(detection_tuple)
                    continue

                h, w = frame.shape[:2]
                x1, y1, x2, y2 = max(0, int(box[0])), max(0, int(box[1])), min(w, int(box[2])), min(h, int(box[3]))
                crop = frame[y1:y2, x1:x2]

                if crop.size > 0:
                    valid_crops.append(crop)
                    valid_indices.append(j)
                    valid_yolo_ids.append(int(ids[j]))

            if valid_crops:
                embeddings = self.reid.to_embeddings(valid_crops)
                proposed_ids, reid_sims, reid_gaps = self.reid.assign_with_scores(
                    embeddings, valid_yolo_ids, update_memory=False
                )

                subset_boxes = np.array([boxes[idx] for idx in valid_indices])

                if self.occlusion_guard:
                    final_ids, alerts = self.occlusion_guard.guard(subset_boxes, proposed_ids, reid_sims, reid_gaps)
                else:
                    final_ids = proposed_ids

                for k, idx in enumerate(valid_indices):
                    tid = int(final_ids[k])
                    box = boxes[idx]
                    x1, y1, x2, y2 = box

                    if tid in self.reid.memory:
                        self.reid._update_existing_id(tid, embeddings[k])
                    else:
                        self.reid._append_embedding(tid, embeddings[k])

                    detection_tuple = (
                        tid,
                        float(x1),
                        float(y1),
                        float(x2 - x1),
                        float(y2 - y1),
                        int(classes[idx])
                    )
                    current_frame_detections.append(detection_tuple)

                    current_frame_players.append({
                        "id": tid,
                        "bbox": box,
                        "cls": int(classes[idx])
                    })

        closest_player_id, closest_team = self._assign_possession(current_ball_pos, current_frame_players)

        self.ball_positions_history.append(current_ball_pos)
        self.players_positions_history.append(current_frame_players)
        self.team_in_possession_history.append(closest_team)

        return current_frame_detections

    def _assign_possession(self, ball_pos, players_data):
        if ball_pos is None or not players_data:
            return None, None

        bx, by = ball_pos
        closest_player = None
        min_dist = float("inf")

        for player in players_data:
            p_cls = int(player["cls"])
            if p_cls == 5:
                continue

            x1, y1, x2, y2 = player["bbox"]
            px = (x1 + x2) / 2
            py = y2

            dist = ((bx - px) ** 2 + (by - py) ** 2) ** 0.5

            if dist < self.possession_threshold and dist < min_dist:
                min_dist = dist
                closest_player = player

        if closest_player is None:
            return None, None

        p_id = closest_player["id"]
        self.possession_dict[p_id] = self.possession_dict.get(p_id, 0) + 1

        team = None
        p_cls = int(closest_player["cls"])
        if p_cls in self.LEFT_CLASSES:
            team = "left"
        elif p_cls in self.RIGHT_CLASSES:
            team = "right"

        if team is not None:
            self.team_possession[team] += 1

        return p_id, team

    def get_possession_stats(self, total_frames=None):
        if total_frames is None:
            total_frames = len(self.ball_positions_history)
        if total_frames == 0:
            return {}
        return {p_id: (count / total_frames) * 100 for p_id, count in self.possession_dict.items()}

    def annotate(self, frames: list[MatLike], detections_list: list, output_path: Path = None, fps: float = 25.0):
        if self.annotator is None:
            print("Warning: Annotator non configurato")
            return frames

        annotated_frames = []

        for i, frame in enumerate(frames):
            if i < len(detections_list):
                frame_copy = frame.copy()

                for detection in detections_list[i]:
                    tid, x, y, w, h, cls = detection
                    box = [x, y, x + w, y + h]

                    if int(cls) == self.BALL_CLASS:
                        self.annotator.highlight_ball(frame_copy, box, tid)
                    else:
                        self.annotator.player_bounding_box(frame_copy, box, tid, 0.31)

                frame_copy = self._draw_possession_overlay(frame_copy, i)
                annotated_frames.append(frame_copy)
            else:
                annotated_frames.append(frame.copy())

        if output_path and annotated_frames:
            duration = len(annotated_frames) / fps
            save_as_video(annotated_frames, duration, output_path, fps)

        return annotated_frames

    def analyze_folder(self, folder_path: Path, output_path: Path = None):
        frames = read_images(folder_path)
        detections = self.process(frames)

        if self.annotator and output_path:
            annotated = self.annotate(frames, detections, output_path)
            return detections, annotated

        return detections, frames

    def analyze_video(self, video_path: Path, output_path: Path = None, fps: float = 25.0):
        temp_folder = Path('/tmp/frames')
        video_to_frames(video_path, temp_folder, fps)
        frames = read_images(temp_folder)
        detections = self.process(frames)

        if self.annotator and output_path:
            annotated = self.annotate(frames, detections, output_path, fps)
            return detections, annotated

        return detections, frames

    def _draw_possession_overlay(self, frame, frame_idx: int):
        hist = self.team_in_possession_history[:frame_idx + 1]
        left_count = sum(1 for t in hist if t == "left")
        right_count = sum(1 for t in hist if t == "right")
        tot = left_count + right_count
    
        if tot == 0:
            left = 0.0
            right = 0.0
        else:
            left = 100.0 * left_count / tot
            right = 100.0 * right_count / tot
    
        team_live = self.team_in_possession_history[frame_idx] if frame_idx < len(self.team_in_possession_history) else None
        if team_live == "left":
            live_txt = "LIVE: LEFT"
        elif team_live == "right":
            live_txt = "LIVE: RIGHT"
        else:
            live_txt = "LIVE: -"
    
        x, y = 15, 15
        w, h = 330, 90
    
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 0), thickness=-1)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), thickness=2)
    
        cv2.putText(frame, "POSSESSION", (x + 12, y + 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
        cv2.putText(frame, f"LEFT  {left:5.1f}%", (x + 12, y + 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"RIGHT {right:5.1f}%", (x + 12, y + 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, live_txt, (x + 190, y + 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
        return frame


In [22]:
class PossessionStats:
    def __init__(self, proximity_threshold: int = 70):
        self.proximity_threshold = proximity_threshold
        self.possession_dict = {} 

    def assign_possession(self, ball_pos, players_data):
        if ball_pos is None or not players_data:
            return None

        bx, by = ball_pos
        closest_player_id = None
        min_dist = float('inf')

        for player in players_data:
            p_id = player['id']
            x1, y1, x2, y2 = player['bbox']
            px, py = (x1 + x2) / 2, (y1 + y2) / 2
            
            dist = ((bx - px)**2 + (by - py)**2)**0.5
            
            if dist < self.proximity_threshold and dist < min_dist:
                min_dist = dist
                closest_player_id = p_id

        if closest_player_id is not None:
            self.possession_dict[closest_player_id] = self.possession_dict.get(closest_player_id, 0) + 1
            
        return closest_player_id

    def get_stats_percentage(self, total_frames):
        if total_frames == 0: return {}
        return {p_id: (count / total_frames) * 100 for p_id, count in self.possession_dict.items()}

In [ ]:
video_to_frames(VIDEO_PATH, IMAGES_FOLDER)

frames = read_images(IMAGES_FOLDER)

In [1]:
%%writefile project_tracker.yaml
tracker_type: botsort
track_high_thresh: 0.15 
track_low_thresh: 0.05  
new_track_thresh: 0.2   
track_buffer: 60        
match_thresh: 0.9       
fuse_score: True        

gmc_method: none        

with_reid: True         
proximity_thresh: 0.4   
appearance_thresh: 0.25 
model: auto

Writing project_tracker.yaml


In [ ]:
print("INIZIALIZZAZIONE")

detector = Detection(yolo)
reid_engine = ReID(reid_model=reid_model, reid_transforms=reid_transforms, 
                   sim_match=0.70, sim_add=0.40)
annotator = Annotator()
swap_guard = OcclusionSwapGuard(iou_cluster=0.2, iou_prev_match=0.3)

processor = VideoProcessor(
    detection=detector,
    reid=reid_engine,
    annotator=annotator,
    occlusion_guard=swap_guard,
    possession_threshold=40
)

print(f" Frame caricati: {len(frames)}")

print("\n  AVVIO ANALISI")
processed_frames = processor.process(frames)
annotated_frames = processor.annotate(frames, processed_frames)
print(f" Analisi completata: {len(processed_frames)} frame")

stats = processor.get_possession_stats(len(frames))
print("\n STATISTICHE POSSESSO INDIVIDUALE:")
for p_id, p_perc in sorted(stats.items(), key=lambda x: x[1], reverse=True):
    print(f"   Giocatore ID {p_id}: {p_perc:.2f}%")

print("\n SALVATAGGIO VIDEO")
save_as_video(annotated_frames, VIDEO_DURATION, VIDEO_OUTPUT)

print("\n" + "="*50)
print(" OPERAZIONE COMPLETATA")
print("="*50)

# Evaluation

## Create Prediction .txt file

In [37]:
KAGGLE_INPUT_GT = Path('/kaggle/input/datasets/robertiacobus/soccernet/test/test')
PRED_ROOT = Path("/kaggle/working/predictions")
OUTPUT_ROOT = Path("/kaggle/working/soccernet_mot_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [35]:
from pathlib import Path
import cv2

def export_mot_predictions(
    test_root: Path,
    pred_root: Path,
    processor,
    include_classes={1, 2, 3, 4},   
    id_offset_per_class=10000,      
    img_ext="jpg"
):
    pred_root.mkdir(parents=True, exist_ok=True)

    seq_paths = sorted([p for p in test_root.iterdir() if p.is_dir() and p.name.startswith("SNMOT-")])

    for seq_path in seq_paths:
        seq_name = seq_path.name
        img_dir = seq_path / "img1"
        img_paths = sorted(img_dir.glob(f"*.{img_ext}"))

        frames = []
        for p in img_paths:
            im = cv2.imread(str(p))
            if im is None:
                raise RuntimeError(f"Immagine non letta: {p}")
            frames.append(im)

        detections_list = processor.process(frames)

        out_path = pred_root / f"{seq_name}.txt"
        with out_path.open("w") as f:
            for frame_idx, dets in enumerate(detections_list, start=1):  
                for det in dets:
                    tid, x, y, w, h, cls = det
                    cls = int(cls)
                    if include_classes is not None and cls not in include_classes:
                        continue

                    tid = int(tid)
                    mot_id = tid + cls * id_offset_per_class

                    conf = 1.0
                    f.write(f"{frame_idx},{mot_id},{x:.2f},{y:.2f},{w:.2f},{h:.2f},{conf:.2f},-1,-1,-1\n")

        print(f"[OK] scritto: {out_path}  (frames: {len(frames)})")


In [ ]:
export_mot_predictions(KAGGLE_INPUT_GT, PRED_ROOT, processor)